In [83]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 재현성을 위한 seed 설정
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [84]:
# 데이터 로드
df = pd.read_csv('../../data/processed/cleaned_disease_dataset_2015landuse.csv')

print(f"Total samples: {len(df):,}")
df.head()

Total samples: 473


,year,country,admin,infection_rate,max_temperature,min_temperature,avg_temperature,Buffa,Cattl,Chick,Ducks,Goats,Horse,Sheep,Swine,rate_landuse,missing_ratio
0,2010,Burkina Faso,Boucle Du Mouhoun,0.585652,307.42114,297.76172,302.50760,0.0,25.000049,164.065700,0.0,34.153420,0.693863,21.774243,9.682960,0.076206,0.0
1,2010,Burkina Faso,Cascades,0.979432,304.47192,297.88160,300.27893,0.0,38.008688,96.864817,0.0,12.506484,0.008877,12.373391,3.865450,0.074387,0.0
2,2010,Burkina Faso,Centre,1.955671,307.73170,299.50660,302.80447,0.0,17.566445,303.908919,0.0,58.134766,0.019041,40.947445,6.821150,0.407856,0.0
3,2010,Burkina Faso,Centre-est,1.037736,307.41138,298.78003,302.61942,0.0,29.597071,203.094681,0.0,69.010943,0.027864,45.871831,18.292361,0.143950,0.0
4,2010,Burkina Faso,Centre-nord,0.379147,308.31274,297.82104,303.14368,0.0,28.843678,138.295650,0.0,65.134086,0.079677,48.346972,4.834737,0.120649,0.0


In [85]:
#자연발생 클러스터링
from jenkspy import JenksNaturalBreaks

df.columns

i = 3

breaks = JenksNaturalBreaks(n_classes=i)
breaks.fit(df["infection_rate"])
print(breaks.breaks_)  # 구간 경계

df["infection_risk_level"] = pd.cut(
    df["infection_rate"], bins=breaks.breaks_, labels=range(i), include_lowest=True
)
df["infection_risk_level"].value_counts()

[0.0, 6.027164685908319, 15.030674846625766, 27.07340324118208]


infection_risk_level
0    340
1     75
2     58
Name: count, dtype: int64

In [86]:
# 컬럼 분류
key_cols = ['year', 'country', 'admin']
target_col = ['infection_rate','infection_risk_level']
climate_cols = ['max_temperature', 'min_temperature', 'avg_temperature']
livestock_cols = ['Buffa', 'Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine']
landuse_cols = ['rate_landuse']
feature_cols = climate_cols + livestock_cols + landuse_cols

print(f"Feature 컬럼 ({len(feature_cols)}개): {feature_cols}")
print(f"Target 컬럼: {target_col}")

Feature 컬럼 (12개): ['max_temperature', 'min_temperature', 'avg_temperature', 'Buffa', 'Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine', 'rate_landuse']
Target 컬럼: ['infection_rate', 'infection_risk_level']


In [87]:
# X, y 분리
X = df[feature_cols].copy()
y = df[target_col].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Feature(X) scaling
X_scaled = X.copy()

# Climate : standard normal dist 
# #         => standard scaling
standard_cols = climate_cols
scaler_climate = StandardScaler()
X_scaled[climate_cols] = scaler_climate.fit_transform(X_scaled[climate_cols])

# Livestock, Landuse : right-skewed dist
#                      => log transformation + standard scaling
skewed_cols = livestock_cols + landuse_cols
for col in skewed_cols:
    X_scaled[col] = np.log1p(X_scaled[col])

scaler_skewed = StandardScaler()
X_scaled[skewed_cols] = scaler_skewed.fit_transform(X_scaled[skewed_cols])

X shape: (473, 12)
y shape: (473, 2)


In [88]:
print("=== Scaling 전후 비교 ===")
# df에서 mean()을 한 번 취하면 컬럼별 평균이, 두 번 취하면 컬럼 평균의 평균(전체 평균)이 구해짐

print("\n[Scaling 전]")
print(f"평균: {X.mean().mean():.4f}, 표준편차: {X.std().mean():.4f}")

print("\n[Scaling 후]")
print(f"평균: {X_scaled.mean().mean():.4f}, 표준편차: {X_scaled.std().mean():.4f}")

=== Scaling 전후 비교 ===

[Scaling 전]
평균: 97.1898, 표준편차: 76.4891

[Scaling 후]
평균: 0.0000, 표준편차: 0.9176


In [89]:
# train, test dataset 분리
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=RANDOM_STATE
)

print("=== Random Split ===")
print(f"Train set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
#print(f"\nTrain - y 평균: {y_train.mean():.4f}, 표준편차: {y_train.std():.4f}")
#print(f"Test - y 평균: {y_test.mean():.4f}, 표준편차: {y_test.std():.4f}")

=== Random Split ===
Train set: 378 samples (79.9%)
Test set: 95 samples (20.1%)


In [90]:
X_test

,max_temperature,min_temperature,avg_temperature,Buffa,Cattl,Chick,Ducks,Goats,Horse,Sheep,Swine,rate_landuse
55,0.290706,0.223722,-0.036723,0.0,-0.287982,0.173418,-0.373720,-0.306537,-0.525160,-0.079111,0.250119,-0.560943
73,-1.096600,-0.168957,-0.804581,0.0,1.729948,0.291567,-0.387296,2.041194,-0.538774,1.455294,2.599255,2.767866
33,0.145823,0.153964,0.231462,0.0,-1.568840,0.031251,1.630063,-1.017555,-0.538774,-1.361643,0.259264,-0.103867
374,0.219202,-0.236829,-0.057478,0.0,-0.126894,0.214707,-0.331094,-0.556304,-0.531796,-1.152476,0.846119,-0.193551
245,-2.366864,-1.289208,-2.027690,0.0,2.017249,1.437211,0.145427,0.903658,-0.529130,0.759724,1.189803,3.281701
...,...,...,...,...,...,...,...,...,...,...,...,...
323,-0.144040,0.814933,0.416888,0.0,0.015895,1.938622,6.274777,0.392441,2.742823,0.784130,1.404432,1.463572
132,1.328118,1.008641,1.234326,0.0,0.859383,0.018359,-0.387296,0.438297,-0.028406,0.370152,-0.764985,-0.314726
460,-0.062204,0.737326,0.402168,0.0,-0.794523,1.802159,-0.387296,0.108141,-0.533212,-0.078040,0.933826,-0.401287
220,-1.531196,-2.735974,-2.205319,0.0,0.977070,0.274080,-0.067552,1.177512,1.639226,1.572120,0.340790,0.559011


In [91]:
y_test

,infection_rate,infection_risk_level
55,1.594533,0
73,1.044634,0
33,8.721241,1
374,17.918314,2
245,7.204301,1
...,...,...
323,2.649538,0
132,0.792752,0
460,4.412564,0
220,23.638778,2


pca

In [92]:
type(X_train)

pandas.core.frame.DataFrame

In [93]:
pca = PCA(n_components= 10)
principalComponents = pca.fit_transform(X_train)
X_train_pcaDf = pd.DataFrame(data=principalComponents, columns = ['principal component1', 'principal component2', '3','4','5','6','7','8','9','10'])
print(pca.explained_variance_ratio_)

X_train_pcaDf

[0.30858278 0.26709626 0.1391547  0.09928429 0.05564953 0.04153926
 0.03214133 0.02294816 0.01981735 0.01322578]


,principal component1,principal component2,3,4,5,6,7,8,9,10
0,-1.361908,0.338825,-2.169809,-0.461508,-0.243074,0.817005,-0.655141,0.530527,-1.764164,0.293988
1,0.211383,2.094929,0.467380,-0.692651,0.376629,0.331493,-0.239140,-0.554830,-0.602953,-0.010677
2,0.030740,-0.930839,0.630018,-0.445570,-0.117385,-0.312753,1.004076,0.748580,-0.064156,0.419534
3,0.997370,2.850826,0.613499,-0.647684,0.897670,0.663495,0.447844,0.660038,-0.566038,-0.310022
4,3.772987,-0.082534,3.160653,-1.022098,-0.146682,-0.961161,-0.447447,0.173087,-0.319957,-0.632838
...,...,...,...,...,...,...,...,...,...,...
373,4.401134,-0.827907,-0.914919,0.193862,-1.642518,-2.572990,-1.178647,0.100790,0.402901,-0.687141
374,-0.416984,-0.777571,-0.459941,-0.540194,-0.937560,-0.063212,-0.862266,1.239215,0.382779,0.934406
375,-3.626943,0.329617,-0.713350,-0.160171,-0.302193,-0.029589,-1.066304,0.529077,-0.043167,-0.385331
376,-0.871584,-0.733671,0.488216,-0.232914,0.246168,-0.065882,0.132918,-0.325237,0.425820,0.210618


pcr 진행

In [94]:
y_train_reset = y_train.reset_index(drop=True)
# PCA 결과(X)와 라벨(y)을 가로로 이어 붙이기
train_finalDf = pd.concat([X_train_pcaDf, y_train_reset], axis=1)

# 합쳐진 데이터 확인 (디버깅용)
print("=== Final DataFrame for Visualization ===")
print(train_finalDf.head())

=== Final DataFrame for Visualization ===
   principal component1  principal component2         3         4         5  \
0             -1.361908              0.338825 -2.169809 -0.461508 -0.243074   
1              0.211383              2.094929  0.467380 -0.692651  0.376629   
2              0.030740             -0.930839  0.630018 -0.445570 -0.117385   
3              0.997370              2.850826  0.613499 -0.647684  0.897670   
4              3.772987             -0.082534  3.160653 -1.022098 -0.146682   

          6         7         8         9        10  infection_rate  \
0  0.817005 -0.655141  0.530527 -1.764164  0.293988        0.694444   
1  0.331493 -0.239140 -0.554830 -0.602953 -0.010677        0.932091   
2 -0.312753  1.004076  0.748580 -0.064156  0.419534       13.944099   
3  0.663495  0.447844  0.660038 -0.566038 -0.310022        1.274313   
4 -0.961161 -0.447447  0.173087 -0.319957 -0.632838        4.628949   

  infection_risk_level  
0                    0  
1     

In [95]:
X_test_pca = pca.transform(X_test)
X_test_pcaDf = pd.DataFrame(data=X_test_pca, columns = ['principal component1', 'principal component2', '3','4','5','6','7','8','9','10'])
y_test_reset = y_test.reset_index(drop = True)

# regression용 y값
y_train_reset_r = y_train_reset['infection_rate'].copy()
y_test_reset_r = y_test_reset['infection_rate'].copy()

# classification용 y값
y_train_reset_c = y_train_reset['infection_risk_level'].copy()
y_test_reset_c = y_test_reset['infection_risk_level'].copy()


test_finalDf = pd.concat([X_test_pcaDf, y_test_reset],axis = 1)

### linear regression(pc 적용)

In [96]:
# linear regression 
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 재현성을 위한 seed 설정
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


model = LinearRegression()
model.fit(X_train_pcaDf, y_train_reset_r)

print("=== Linear Regression 모델 학습 완료 ===")
print(f"Intercept (절편): {model.intercept_:.4f}")
print(f"Number of features: {len(model.coef_)}")



=== Linear Regression 모델 학습 완료 ===
Intercept (절편): 5.5219
Number of features: 10


In [97]:
# 평가 지표 계산
def evaluate_model(y_true, y_pred, dataset_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    # MAPE: 0인 값 제외
    mask = y_true > 0
    if mask.sum() > 0:
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    else:
        mape = np.inf
    
    print(f"\n=== {dataset_name} Set Performance (원본 스케일) ===")
    print(f"RMSE (Root Mean Squared Error): {rmse:.4f}")
    print(f"MAE (Mean Absolute Error): {mae:.4f}")
    print(f"R² Score: {r2:.4f}")
    print(f"MAPE (Mean Absolute Percentage Error): {mape:.2f}%")
    
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': mape}

train_pred = model.predict(X_train_pcaDf)
test_pred = model.predict(X_test_pcaDf)

train_metrics = evaluate_model(y_train_reset_r, train_pred, "Train")
test_metrics = evaluate_model(y_test_reset_r, test_pred, "Test")


=== Train Set Performance (원본 스케일) ===
RMSE (Root Mean Squared Error): 4.4108
MAE (Mean Absolute Error): 3.3296
R² Score: 0.5662
MAPE (Mean Absolute Percentage Error): 212.33%

=== Test Set Performance (원본 스케일) ===
RMSE (Root Mean Squared Error): 3.7276
MAE (Mean Absolute Error): 2.9549
R² Score: 0.6319
MAPE (Mean Absolute Percentage Error): 203.78%


linear regression에서는 기본 피처보다 pca 피처를 도입했을 때 R2 score가 조금 더 높아짐

--- 기본 피처 performance ---
1. Train Set Performance (원본 스케일) 
- RMSE (Root Mean Squared Error): 5.1506
- MAE (Mean Absolute Error): 3.2286
- R² Score: 0.4085
- MAPE (Mean Absolute Percentage Error): 124.96%

2. Test Set Performance (원본 스케일)
- RMSE (Root Mean Squared Error): 4.8878
- MAE (Mean Absolute Error): 3.2662
- R² Score: 0.3670
- MAPE (Mean Absolute Percentage Error): 156.91%


--- pca 피처 performance ---
1. Train Set Performance
- RMSE (Root Mean Squared Error): 4.4108
- MAE (Mean Absolute Error): 3.3296
- R² Score: 0.5662
- MAPE (Mean Absolute Percentage Error): 212.33%

2. Test Set Performance
- RMSE (Root Mean Squared Error): 3.7276
- MAE (Mean Absolute Error): 2.9549
- R² Score: 0.6319
- MAPE (Mean Absolute Percentage Error): 203.78%

### random forest(pc 적용)

In [99]:
#  랜덤포레스트 
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


rf = RandomForestClassifier(random_state = 0)
rf.fit(X_train_pcaDf, y_train_reset_c)
pred = rf.predict(X_test_pcaDf)
accuracy = accuracy_score(y_test_reset_c, pred)
print('pca 랜덤 포레스트 정확도: {:.4f}'.format(accuracy))

pca 랜덤 포레스트 정확도: 0.8421


In [ ]:
# Accuracy 분석

from sklearn.metrics import classification_report, accuracy_score

print("Complaint Level Classification Accuracy:", round(accuracy_score(y_test_reset_c, pred), 3))
print(classification_report(y_test_reset_c, pred, target_names=['Low', 'Mid', 'High']))

Complaint Level (Kmeans) Classification Accuracy: 0.842
              precision    recall  f1-score   support

         Low       0.87      0.99      0.92        67
         Mid       0.73      0.40      0.52        20
        High       0.75      0.75      0.75         8

    accuracy                           0.84        95
   macro avg       0.78      0.71      0.73        95
weighted avg       0.83      0.84      0.82        95



--- 기본 랜덤포레스트 성능 ---
- 자연발생 클러스터링 기반 랜덤 포레스트 정확도: 0.8316
- Complaint Level Classification Accuracy: 0.832

                    precision    recall  f1-score   support

         Low             0.87      0.97      0.92        67
         Mid             0.67      0.40      0.50        20
        High             0.75      0.75      0.75         8

        accuracy           -         -       0.83        95
        macro avg        0.76      0.71      0.72        95
        weighted avg     0.81      0.83      0.81        95

In [103]:
from imblearn.over_sampling import SMOTE

# 1. 가짜 데이터를 생성해서 Mid, High 개수를 Low만큼 뻥튀기합니다.
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_pcaDf, y_train_reset_c)

# 2. 뻥튀기된 데이터로 학습 (가중치 옵션 없이도 데이터가 많아서 잘 배움)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_resampled, y_train_resampled) # resampled 데이터 사용!

# 3. 평가
pred = rf.predict(X_test_pcaDf)

print("Complaint Level Classification Accuracy:", round(accuracy_score(y_test_reset_c, pred), 3))
print(classification_report(y_test_reset_c, pred, target_names=['Low', 'Mid', 'High']))

Complaint Level Classification Accuracy: 0.842
              precision    recall  f1-score   support

         Low       0.91      0.94      0.93        67
         Mid       0.69      0.55      0.61        20
        High       0.60      0.75      0.67         8

    accuracy                           0.84        95
   macro avg       0.73      0.75      0.73        95
weighted avg       0.84      0.84      0.84        95



정리

- linear regression에서는 pca 버전이 성능 향상을 보임
    - Linear Regression은 다중공선성(변수 간 상관관계)에 취약한데, PCA가 이를 해결해주니 성능이 올랐을 수도 있음..
- random forest에서는 Pca 버전이 성능 차이를 가져오지 못함
    - Random Forest는 원래 변수 간의 상호작용을 잘 잡아내고 공선성에 강하기 때문에, PCA로 정보가 압축(손실)되면 오히려 성능이 정체되거나 떨어질 수 있음..